# EXEMPLO 2 — BGE-Reranker: Reranking com Cross-Encoder
## MBA RAG & CAG Aplicados a Direito e Segurança Pública — Aula 3

**Objetivo:** Demonstrar o uso do BGE-Reranker-v2-m3 para reordenar resultados de retrieval.

**O que você vai aprender:**
1. Diferença prática entre bi-encoder (busca) e cross-encoder (reranking)
2. Como carregar e usar o BGE-Reranker
3. Interpretar scores de relevância
4. Visualizar o impacto do reranking nos resultados

> ⚠️ Este é um exemplo de **referência rápida**. O lab completo com pipeline integrado está em `labs/LAB2_BGE_Reranker.ipynb`.


## ⚙️ 1. Instalação

In [ ]:
!pip install transformers torch sentence-transformers pandas -q
print("✅ Dependências instaladas")

## 📚 2. Contexto: Por Que Reranking?

In [ ]:
# Simulação: resultado de busca vetorial (bi-encoder)
# Imagine que buscamos: "O suspeito pode ficar calado no interrogatório?"

query = "O suspeito pode ficar calado no interrogatório?"

# Estes são os top-5 retornados pelo retrieval vetorial (antes do reranking)
# Ordenados por similaridade de cosseno com a query
resultados_retrieval = [
    {
        "rank_original": 1,
        "score_coseno": 0.892,
        "texto": "O interrogatório é o ato pelo qual o juiz ouve o acusado sobre a imputação que lhe é feita. Trata-se de meio de defesa, não de prova. O acusado não é obrigado a responder perguntas.",
        "fonte": "CPP Art. 186 comentado"
    },
    {
        "rank_original": 2,
        "score_coseno": 0.871,
        "texto": "No interrogatório o juiz informará o réu do teor da acusação. Perguntará se tem advogado constituído ou se deseja que lhe seja nomeado defensor público.",
        "fonte": "CPP Art. 187"
    },
    {
        "rank_original": 3,
        "score_coseno": 0.854,
        "texto": "É assegurado ao preso o direito de permanecer calado. Ser informado de seus direitos. Não ser obrigado a produzir prova contra si mesmo.",
        "fonte": "CF/88 Art. 5º LXIII"
    },
    {
        "rank_original": 4,
        "score_coseno": 0.831,
        "texto": "O silêncio do acusado não importará confissão, mas poderá constituir elemento para a formação do convencimento do juiz.",
        "fonte": "CPP Art. 186 parágrafo único"
    },
    {
        "rank_original": 5,
        "score_coseno": 0.819,
        "texto": "Confissão obtida sem advertência do direito ao silêncio é prova ilícita. O investigado deve ser informado de seus direitos antes de qualquer declaração.",
        "fonte": "STJ — RHC 99.735/MG"
    }
]

print("🔍 RESULTADO DO RETRIEVAL VETORIAL (antes do reranking)")
print("=" * 65)
print(f"Query: '{query}'")
print()
for r in resultados_retrieval:
    print(f"  #{r['rank_original']} [coseno: {r['score_coseno']:.3f}] {r['fonte']}")
    print(f"      {r['texto'][:80]}...")
    print()
print("❓ Qual chunk realmente responde MELHOR à pergunta?")
print("   O bi-encoder classifica por similaridade global, não por relevância específica.")

## 🤖 3. Carregando o BGE-Reranker

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import numpy as np

MODEL_NAME = "BAAI/bge-reranker-v2-m3"

print(f"⏳ Carregando {MODEL_NAME}...")
print("   (1ª execução faz download ~560MB — aguarde)")

try:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
    model.eval()
    
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = model.to(device)
    
    print(f"✅ Modelo carregado! Device: {device}")
    print(f"   Parâmetros: {sum(p.numel() for p in model.parameters()):,}")
    USE_REAL_MODEL = True
    
except Exception as e:
    print(f"⚠️ Modelo não disponível ({e})")
    print("   Usando scores simulados para demonstração")
    USE_REAL_MODEL = False

## 📐 4. Função de Reranking

In [ ]:
def rerank_documents(query: str, documents: list, batch_size: int = 8) -> list:
    """
    Reordena documentos usando cross-encoder BGE-Reranker.
    
    Args:
        query: Query original do usuário
        documents: Lista de dicts com campo 'texto'
        batch_size: Tamanho do batch para processamento
    
    Returns:
        Lista de documentos reordenados com campo 'score_reranker'
    """
    if not USE_REAL_MODEL:
        # Scores simulados para demonstração
        # Na prática, o cross-encoder vê TODA a query + documento juntos
        simulated_scores = [0.94, 0.71, 0.96, 0.88, 0.91]
        results = []
        for doc, score in zip(documents, simulated_scores):
            results.append({**doc, "score_reranker": score})
        return sorted(results, key=lambda x: x["score_reranker"], reverse=True)
    
    texts = [doc["texto"] for doc in documents]
    pairs = [[query, text] for text in texts]
    
    all_scores = []
    for i in range(0, len(pairs), batch_size):
        batch = pairs[i:i + batch_size]
        inputs = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors="pt"
        ).to(device)
        
        with torch.no_grad():
            logits = model(**inputs).logits.squeeze(-1)
            scores = torch.sigmoid(logits).cpu().numpy()
        
        all_scores.extend(scores.tolist())
    
    results = []
    for doc, score in zip(documents, all_scores):
        results.append({**doc, "score_reranker": round(score, 4)})
    
    return sorted(results, key=lambda x: x["score_reranker"], reverse=True)

print("✅ Função de reranking definida")
print()
print("📖 Como funciona internamente:")
print("   Input: [CLS] query [SEP] documento [SEP]")
print("   O transformer processa JUNTOS todos os tokens")
print("   Output: sigmoid(W · h_CLS) → score entre 0 e 1")

## 🔄 5. Aplicando o Reranking

In [ ]:
import time

print("⏳ Aplicando BGE-Reranker...")
start = time.time()

resultados_reranked = rerank_documents(query, resultados_retrieval)

elapsed = time.time() - start
print(f"✅ Reranking concluído em {elapsed:.2f}s")
print()
print("🏆 RESULTADO APÓS RERANKING")
print("=" * 65)
print(f"Query: '{query}'")
print()
for i, r in enumerate(resultados_reranked, 1):
    mudanca = r['rank_original'] - i
    seta = f"↑{abs(mudanca)}" if mudanca > 0 else (f"↓{abs(mudanca)}" if mudanca < 0 else "─")
    print(f"  #{i} [{seta}] [reranker: {r['score_reranker']:.3f}] {r['fonte']}")
    print(f"      {r['texto'][:80]}...")
    print()

## 📊 6. Comparação Visual

In [ ]:
import pandas as pd

print("📊 COMPARAÇÃO: Antes vs Depois do Reranking")
print("=" * 75)

comparison = []
for r in resultados_reranked:
    comparison.append({
        "Rank Original": r['rank_original'],
        "Rank Final": resultados_reranked.index(r) + 1,
        "Score Coseno": r['score_coseno'],
        "Score Reranker": r['score_reranker'],
        "Fonte": r['fonte']
    })

df = pd.DataFrame(comparison).sort_values("Rank Final")
print(df.to_string(index=False))
print()

# Análise da mudança mais importante
mais_subiu = max(comparison, key=lambda x: x['Rank Original'] - x['Rank Final'])
print(f"🔝 Maior subida: '{mais_subiu['Fonte']}'")
print(f"   {mais_subiu['Rank Original']}º → {mais_subiu['Rank Final']}º")
print()
print("💡 INSIGHT: O chunk do Art. 5º LXIII (direito constitucional ao silêncio)")
print("   sobe no ranking porque o cross-encoder identifica que ele responde")
print("   DIRETAMENTE à pergunta 'pode ficar calado', enquanto o Art. 187")
print("   trata de procedimentos do interrogatório — relacionado, mas menos relevante.")

## 🎯 7. Interpretação dos Scores

In [ ]:
print("📏 GUIA DE INTERPRETAÇÃO DOS SCORES DO RERANKER")
print("=" * 55)
print()

score_guide = [
    ("0.85 – 1.00", "Altamente relevante", "✅ Incluir no contexto"),
    ("0.60 – 0.84", "Provavelmente relevante", "✅ Incluir se espaço disponível"),
    ("0.30 – 0.59", "Relevância moderada", "⚠️  Incluir com cautela"),
    ("0.00 – 0.29", "Baixa relevância", "❌ Descartar"),
]

for score_range, interpretacao, acao in score_guide:
    print(f"  {score_range:15s} | {interpretacao:25s} | {acao}")

print()
print("📌 Aplicação no nosso caso:")
for r in resultados_reranked:
    rank = resultados_reranked.index(r) + 1
    score = r['score_reranker']
    if score >= 0.85:
        status = "✅ Incluir"
    elif score >= 0.60:
        status = "✅ Incluir"
    elif score >= 0.30:
        status = "⚠️  Cautela"
    else:
        status = "❌ Descartar"
    print(f"  #{rank} [{score:.3f}] {status} — {r['fonte']}")

print()
print("✅ Exemplo concluído! Veja o LAB2 para pipeline integrado com OpenSearch.")

## 📚 Referências
- Nogueira, R. & Cho, K. (2019). *Passage Re-ranking with BERT*. arXiv:1901.04085
- Chen, J. et al. (2024). *BGE M3-Embedding*. arXiv:2309.07597
- BAAI/bge-reranker-v2-m3: https://huggingface.co/BAAI/bge-reranker-v2-m3
